# 🔍 Real-Time Fraud Detection — EDA & Model Training
**Dataset**: [Kaggle Credit Card Fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Model**: XGBoost with SMOTE + SHAP Explainability

## 1️⃣ Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, precision_recall_curve,
                             average_precision_score, f1_score)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import joblib
import os

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('✅ Libraries loaded!')

## 2️⃣ Load Dataset
> Download from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
> Place `creditcard.csv` in the `data/` folder

In [ ]:
DATA_PATH = '../data/creditcard.csv'

# ── If you don't have the file yet, generate synthetic sample data ──
if not os.path.exists(DATA_PATH):
    print('⚠️  creditcard.csv not found — generating synthetic demo data...')
    np.random.seed(42)
    n = 10000
    # 28 PCA features V1–V28 + Time + Amount + Class
    V_cols = {f'V{i}': np.random.randn(n) for i in range(1, 29)}
    df = pd.DataFrame(V_cols)
    df['Time']   = np.random.uniform(0, 172800, n)
    df['Amount'] = np.abs(np.random.exponential(88, n))
    # Inject 2% fraud — make frauds slightly separable
    fraud_idx = np.random.choice(n, size=200, replace=False)
    df['Class'] = 0
    df.loc[fraud_idx, 'Class'] = 1
    df.loc[fraud_idx, 'V1'] -= 3
    df.loc[fraud_idx, 'V3'] += 2
    df.to_csv(DATA_PATH, index=False)
    print(f'✅ Synthetic data saved → {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 3️⃣ Exploratory Data Analysis

In [ ]:
# ── Basic Info ──
print('=== Dataset Info ===')
print(f'Total transactions : {len(df):,}')
print(f'Fraudulent         : {df["Class"].sum():,} ({df["Class"].mean()*100:.3f}%)')
print(f'Legitimate         : {(df["Class"]==0).sum():,}')
print(f'Missing values     : {df.isnull().sum().sum()}')
print()
df.describe()

In [ ]:
# ── Class Imbalance ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df['Class'].value_counts()
axes[0].pie(counts, labels=['Legitimate', 'Fraud'], autopct='%1.2f%%',
            colors=['#4CAF50', '#F44336'], startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')

# Amount distribution by class
df[df['Class']==0]['Amount'].hist(bins=60, ax=axes[1], alpha=0.6, color='#4CAF50', label='Legitimate')
df[df['Class']==1]['Amount'].hist(bins=60, ax=axes[1], alpha=0.7, color='#F44336', label='Fraud')
axes[1].set_xlabel('Transaction Amount (€)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Amount Distribution by Class', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].set_xlim(0, 1000)

plt.tight_layout()
plt.savefig('../data/eda_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('💡 Key Insight: Massive class imbalance → need SMOTE!')

In [ ]:
# ── Feature Correlation with Fraud ──
correlations = df.corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#F44336' if x > 0 else '#2196F3' for x in correlations]
correlations.plot(kind='bar', ax=ax, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Fraud Class', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Pearson Correlation')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/eda_correlations.png', dpi=120, bbox_inches='tight')
plt.show()
print('Top positive correlations (fraud indicators):')
print(correlations.tail(5))

In [ ]:
# ── Top features violin plots ──
top_features = correlations.abs().sort_values(ascending=False).index[:6].tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    data_plot = [
        df[df['Class']==0][feat].values,
        df[df['Class']==1][feat].values
    ]
    parts = axes[i].violinplot(data_plot, showmedians=True)
    for j, pc in enumerate(parts['bodies']):
        pc.set_facecolor('#4CAF50' if j==0 else '#F44336')
        pc.set_alpha(0.7)
    axes[i].set_title(feat, fontsize=12, fontweight='bold')
    axes[i].set_xticks([1, 2])
    axes[i].set_xticklabels(['Legit', 'Fraud'])
    axes[i].set_ylabel('Value')

plt.suptitle('Top Feature Distributions: Legit vs Fraud', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/eda_violin.png', dpi=120, bbox_inches='tight')
plt.show()

## 4️⃣ Feature Engineering & Preprocessing

In [ ]:
# ── Scale Amount (Time is dropped — not predictive) ──
df_processed = df.drop(columns=['Time']).copy()

scaler = StandardScaler()
df_processed['Amount'] = scaler.fit_transform(df_processed[['Amount']])

X = df_processed.drop('Class', axis=1)
y = df_processed['Class']

print(f'Features: {X.shape[1]}')
print(f'Class balance before SMOTE: {y.value_counts().to_dict()}')

In [ ]:
# ── Train/Test Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Apply SMOTE on training set only ──
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE → Train: {dict(pd.Series(y_train).value_counts())}')
print(f'After  SMOTE → Train: {dict(pd.Series(y_train_res).value_counts())}')
print(f'Test set (no SMOTE): {dict(pd.Series(y_test).value_counts())}')

## 5️⃣ Model Training — XGBoost

In [ ]:
# ── Train XGBoost ──
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test, y_test)],
    verbose=50
)
print('\n✅ Model trained!')

## 6️⃣ Model Evaluation

In [ ]:
# ── Predictions ──
y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# ── Metrics ──
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc  = average_precision_score(y_test, y_pred_proba)
f1      = f1_score(y_test, y_pred)
print(f'ROC-AUC Score        : {roc_auc:.4f}')
print(f'PR-AUC Score         : {pr_auc:.4f}')
print(f'F1 Score (Fraud)     : {f1:.4f}')

In [ ]:
# ── Confusion Matrix ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'], ax=axes[0],
            linewidths=1, linecolor='white')
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Precision-Recall Curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
axes[1].plot(recall, precision, color='#E91E63', linewidth=2.5, label=f'PR-AUC = {pr_auc:.4f}')
axes[1].fill_between(recall, precision, alpha=0.1, color='#E91E63')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/evaluation_plots.png', dpi=120, bbox_inches='tight')
plt.show()

## 7️⃣ SHAP Explainability

In [ ]:
# ── Compute SHAP Values ──
print('Computing SHAP values...')
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# ── Global Feature Importance ──
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('SHAP Global Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/shap_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Beeswarm (how each feature affects fraud score) ──
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Summary — Feature Impact on Fraud Probability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Waterfall for ONE fraud case (explainability for single transaction) ──
fraud_indices = np.where(y_test == 1)[0]
if len(fraud_indices) > 0:
    sample_idx = fraud_indices[0]
    print(f'Explaining fraud transaction at test index: {sample_idx}')
    explanation = explainer(X_test.iloc[[sample_idx]])
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(explanation[0], show=False)
    plt.title('Why was this transaction flagged as FRAUD?', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/shap_waterfall_fraud.png', dpi=120, bbox_inches='tight')
    plt.show()

## 8️⃣ Save Model Artifacts

In [ ]:
os.makedirs('../model', exist_ok=True)

# Save model
joblib.dump(model,   '../model/xgb_fraud_model.pkl')
# Save scaler
joblib.dump(scaler,  '../model/amount_scaler.pkl')
# Save feature names
joblib.dump(list(X.columns), '../model/feature_names.pkl')
# Save test sample for demo
X_test.to_csv('../model/test_sample.csv', index=False)

print('✅ Model artifacts saved:')
print('   → model/xgb_fraud_model.pkl')
print('   → model/amount_scaler.pkl')
print('   → model/feature_names.pkl')
print('   → model/test_sample.csv')
print(f'\nFinal ROC-AUC: {roc_auc:.4f}')
print(f'Final PR-AUC : {pr_auc:.4f}')